## MIMO Problem Analysis


In [14]:
import control as ct
import numpy as np

A = np.zeros((6,6))
B = np.zeros((6,2))
C = np.zeros((3,6))
D = np.zeros((3,2))

A[0][3] = 1
A[1][4] = 1
A[2][5] = 1
A[5][1] = -9.8

B[3][0] = 2
B[4][1] = 1/6

C[0][0] = 1
C[1][1] = 1
C[2][2] = 1

sys = ct.ss(A,B,C,D)
poles = ct.poles(sys)

This is currently an unstable system so we need to close the loop and make this a stable system. I will use LQR to find the matrix K that makes this system stable and on the left hand side.

In [15]:
Q = np.identity(6)
R = np.identity(2)

K, S, E = ct.lqr(A, B, Q, R)
print(K)
print(S)
print(E)

full_state_sys = ct.ss(A - B @ K,B,C,D)

[[ 1.00000000e+00  3.76530069e-15  7.38727476e-17  1.41421356e+00
   3.66846833e-15 -1.97958132e-16]
 [ 3.55569750e-16  2.92003852e+01 -1.00000000e+00  3.05705694e-16
   1.87457895e+01 -2.63804137e+00]]
[[ 1.41421356e+00  5.23999560e-15 -1.75937527e-16  5.00000000e-01
   2.13341850e-15 -5.44080794e-16]
 [ 5.23999560e-15  3.92267440e+02 -1.87457895e+01  1.88265034e-15
   1.75202311e+02 -4.34521682e+01]
 [-1.75937527e-16 -1.87457895e+01  2.63804137e+00  3.69363738e-17
  -6.00000000e+00  2.97963114e+00]
 [ 5.00000000e-01  1.88265034e-15  3.69363738e-17  7.07106781e-01
   1.83423416e-15 -9.89790662e-17]
 [ 2.13341850e-15  1.75202311e+02 -6.00000000e+00  1.83423416e-15
   1.12474737e+02 -1.58282482e+01]
 [-5.44080794e-16 -4.34521682e+01  2.97963114e+00 -9.89790662e-17
  -1.58282482e+01  5.94755457e+00]]
[-1.41421351+0.j         -1.41421361+0.j         -0.56611477+1.11902132j
 -0.56611477-1.11902132j -0.99603435+0.21557543j -0.99603435-0.21557543j]


Now we compute the Grammians of the system with the matrix K

In [16]:
from scipy.linalg import solve_continuous_are, solve_continuous_lyapunov

P_riccati = solve_continuous_are(A,B,Q,R)

K = np.linalg.inv(R) @ (B.T @ P_riccati)

A_cl = A - B @ K
print(A_cl)

[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   1.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  1.00000000e+00]
 [-2.00000000e+00 -7.53060138e-15 -1.47745495e-16 -2.82842712e+00
  -7.33693666e-15  3.95916265e-16]
 [-5.92616250e-17 -4.86673087e+00  1.66666667e-01 -5.09509490e-17
  -3.12429824e+00  4.39673562e-01]
 [ 0.00000000e+00 -9.80000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00]]


In [17]:
gram_c = solve_continuous_lyapunov(A_cl, -B @ B.T)
gram_o = solve_continuous_lyapunov(A_cl.T, -C.T @ C)

In [18]:
eigenvalues, eigenvectors = np.linalg.eig(gram_c)
print(eigenvalues)
print(eigenvectors)

#most controllable is eigenvector of max eigenvalue
#opposite is true

[0.35355339 0.70710678 0.28764977 0.13715628 0.00127172 0.00435628]
[[ 1.00000000e+00  3.41890886e-16  3.48744481e-16 -3.97421344e-16
  -1.71643999e-17 -7.86941949e-19]
 [ 0.00000000e+00 -3.01317828e-16 -4.79457872e-02 -2.54299105e-17
  -9.98849939e-01 -1.26684850e-15]
 [ 0.00000000e+00  3.65702267e-16 -9.98849939e-01 -8.12418483e-16
   4.79457872e-02 -3.34317253e-16]
 [ 0.00000000e+00 -1.00000000e+00 -3.66305619e-16  7.41238199e-17
  -3.66996072e-17  2.98571534e-17]
 [ 0.00000000e+00  5.09571361e-17  1.75203975e-16  1.43926824e-01
   4.85052114e-15 -9.89588333e-01]
 [ 0.00000000e+00  3.50924054e-16 -1.53351688e-15  9.89588333e-01
  -5.91156371e-16  1.43926824e-01]]


According to the max eigenvalue which is 0.707 in the 2nd index starting at 1. We can look at the 2nd column of the eigenvectors matrix and see the largest magnitude being -1.00. This position of the -1.0 relates to the $\dot{z}$ which is thrust which makes sense. This means that the most reachable state is the verticle velocity.

Now we need to calculate the Hankel Singular Values

In [19]:
hsvs = np.sqrt(np.linalg.eigvals(gram_c @ gram_o))

print(hsvs)

[0.60355339 0.10355339 0.77870023 0.36952086 0.10310698 0.01510489]


In [20]:
from scipy import linalg

# 1. Get the Cholesky factor of your Controllability Grammian
L = linalg.cholesky(gram_c, lower=True)

# 2. Singular Value Decomposition of the observability-weighted factor
U, s, Vh = linalg.svd(L.T @ gram_o @ L)

# 3. Construct the transformation matrix T and its inverse
# Note: 's' here are the squares of the Hankel Singular Values
Sigma_sqrt = np.diag(np.sqrt(np.sqrt(s))) 
T = Sigma_sqrt @ U.T @ linalg.inv(L)
T_inv = linalg.inv(T)

# 4. Transform your original (Open-Loop) matrices into Balanced Form
A_bal = T @ A @ T_inv
B_bal = T @ B
C_bal = C @ T_inv
D_bal = D # D remains the same

In [21]:

# 1. Start with your Balanced Realization matrices from Step 3
# A_bal, B_bal, C_bal, D_bal (6 states, 2 inputs, 3 outputs)

# 2. Define how the Target (w) affects the system
# Since 'w' is a reference for the first state (altitude z), 
# B1 maps the target to the state dynamics (usually zero if the target is just for error calc)
B1 = np.zeros((6, 1)) 

# 3. Define the Regulated Output (z_reg = Actual Altitude - Target)
# C1 picks out 'z' from the balanced states
# D11 maps the Target (w) directly to the error (usually -1)
# D12 maps the Control (u) to the error (usually 0)
C1 = C_bal[0:1, :]   # Pulls the altitude row from the balanced C matrix
D11 = np.array([[-1]]) 
D12 = np.zeros((1, 2))

# 4. Define the Measured Output (y) mapping
# D21 maps Target (w) to sensors (usually 0)
# D22 is the direct feedthrough of the plant (usually 0)
D21 = np.zeros((3, 1))
D22 = D_bal

# 5. Assemble the Four-Block Form (The "Augmented Plant" P)
# This represents the mapping from [w; u] to [z_reg; y]
A_aug = A_bal
B_aug = np.hstack([B1, B_bal])
C_aug = np.vstack([C1, C_bal])
D_aug = np.block([
    [D11, D12],
    [D21, D22]
])

print("Four-Block Realization complete.")

Four-Block Realization complete.


In [ ]:
print(A_aug.shape)
print(D_aug.shape)

print(np.linalg.eigvals(A_aug))

(6, 6)
(4, 3)
